# Macro Plots en Python + Plotly

Notebook simple centrado en tres pasos:
1. Ingesta del dataset generado por el proyecto.
2. Proceso para llevarlo a formato tabular.
3. Visualización con Plotly replicando la lógica del dashboard.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

DATA_PATH = Path('../public/data/commodities.json')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'No existe {DATA_PATH}. Ejecuta primero `npm run build:data`.')

with DATA_PATH.open() as f:
    payload = json.load(f)

payload.keys()

## 1. Ingesta

Leemos el `commodities.json` ya normalizado por el proyecto. Cada commodity trae:
- metadatos
- summary
- series por año
- puntos diarios con `day`, `date`, `price` y `changePct`

In [ ]:
for key in ['oil', 'gold', 'sp500']:
    dataset = payload[key]
    print(f"{dataset['name']}: {len(dataset['series'])} años, currentYear={dataset['summary']['currentYear']}, generatedAt={payload['generatedAt']}")

## 2. Proceso

Convertimos las series anuales a un `DataFrame` largo. Esto deja listo el dataset para filtrar, resumir y graficar.

In [ ]:
def dataset_to_frame(dataset: dict) -> pd.DataFrame:
    rows = []
    current_year = dataset['summary']['currentYear']

    for year_entry in dataset['series']:
        year = year_entry['year']
        is_current = year == current_year
        for point in year_entry['points']:
            rows.append({
                'commodity_key': dataset['key'],
                'commodity_name': dataset['name'],
                'unit': dataset['unit'],
                'year': year,
                'is_current': is_current,
                'day': point['day'],
                'date': pd.to_datetime(point['date']),
                'price': point['price'],
                'change_pct': point['changePct'],
            })

    return pd.DataFrame(rows)


frames = [dataset_to_frame(payload[key]) for key in ['oil', 'gold', 'sp500']]
df = pd.concat(frames, ignore_index=True)
df.head()

In [ ]:
summary = (
    df.groupby(['commodity_name', 'year', 'is_current'], as_index=False)
      .agg(last_day=('day', 'max'), last_change_pct=('change_pct', 'last'))
)
summary.tail(10)

## 3. Gráfico

La lógica visual replica el dashboard:
- años históricos en gris
- año actual destacado en dorado
- eje X por trading day
- eje Y por cambio YTD (%)

In [ ]:
accent = '#ffd166'
historical = 'rgba(255,255,255,0.20)'
panel_bg = '#050505'
paper_bg = '#000000'
font_color = '#f2f2f2'

datasets = [payload['oil'], payload['gold'], payload['sp500']]
fig = make_subplots(
    rows=3,
    cols=1,
    vertical_spacing=0.08,
    subplot_titles=[dataset['name'] for dataset in datasets],
)

for row_index, dataset in enumerate(datasets, start=1):
    data = df[df['commodity_key'] == dataset['key']].copy()
    current_year = dataset['summary']['currentYear']

    for year, year_df in data.groupby('year', sort=True):
        is_current = year == current_year
        fig.add_trace(
            go.Scatter(
                x=year_df['day'],
                y=year_df['change_pct'],
                mode='lines',
                name=str(year),
                legendgroup=dataset['key'],
                showlegend=is_current,
                line=dict(
                    color=accent if is_current else historical,
                    width=3 if is_current else 1,
                ),
                hovertemplate=(
                    '<b>%{fullData.name}</b><br>'
                    'Trading day %{x}<br>'
                    'YTD %{y:.2f}%<br>'
                    'Price %{customdata[0]:,.2f}<br>'
                    'Date %{customdata[1]|%Y-%m-%d}'
                    '<extra></extra>'
                ),
                customdata=year_df[['price', 'date']].to_numpy(),
            ),
            row=row_index,
            col=1,
        )

    fig.update_xaxes(
        title_text='Trading Days',
        showgrid=False,
        zeroline=False,
        row=row_index,
        col=1,
    )
    fig.update_yaxes(
        title_text='YTD Change (%)',
        gridcolor='rgba(255,255,255,0.09)',
        zeroline=False,
        ticksuffix='%',
        row=row_index,
        col=1,
    )

fig.update_layout(
    height=1100,
    paper_bgcolor=paper_bg,
    plot_bgcolor=panel_bg,
    font=dict(color=font_color, family='Arial, sans-serif', size=11),
    title=dict(text='Commodity Dashboard: YTD by Trading Day', x=0.02),
    margin=dict(l=50, r=25, t=70, b=40),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
)

display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

## Observación

Este notebook está hecho en Python de verdad: la ingesta, el procesamiento y el gráfico salen de Python. Si después quieres, puedo hacer una segunda versión más analítica con:
- percentiles históricos
- banda min/max por día
- tablas resumen por commodity